# 🎨 GÁN NHÃN supervision — MÀU + TÊN NHÃN + QUỸ ĐẠO · 1 bài toán, 2 video

**Bài toán: đếm/theo dõi PHƯƠNG TIỆN** trên **2 video** (giao lộ + cao tốc). Dùng
**YOLOv8 + supervision** để gán nhãn đẹp:
- 🎨 **MÀU theo track** — mỗi xe một màu riêng (box + nhãn + vệt cùng màu).
- 🏷️ **TÊN NHÃN** — loại xe + mã track: `car #3`, `truck #7`, `bus #12`.
- 🌀 **QUỸ ĐẠO di chuyển** — vệt (trace) cho thấy đường xe đã đi.

Xuất **video annotate + ảnh** cho cả 2 video. Cần **GPU T4**.

## 1) Cài đặt + tải code

In [ ]:
import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
REPO = os.path.join(WORK, "VisionOS"); BR = "claude/locate-anything-test-suite-xwju2f"
if not os.path.isdir(os.path.join(REPO, ".git")):
    os.system(f"git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git {REPO}")
os.chdir(REPO)
os.system(f"git fetch -q origin {BR} && git checkout -q {BR} && git reset --hard -q origin/{BR}")
os.chdir(os.path.join(REPO, "VisionOS"))
os.system("pip install -q ultralytics 'supervision>=0.21' opencv-python-headless")
import torch
print("📁", os.getcwd(), "| 🖥️", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "❌ CHƯA BẬT GPU (Colab: Runtime→T4 · Kaggle: Settings→Accelerator→GPU)")

## 2) Nạp YOLOv8 (recall cao) + dựng ANNOTATOR supervision
`ColorLookup.TRACK` = tô màu theo **track-id** → mỗi xe một màu xuyên suốt.

In [ ]:
import sys, subprocess, warnings
warnings.simplefilter("ignore")
import numpy as np, cv2, supervision as sv
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
from recognition.detectors import load_standard_detector
from recognition.sv_counting import _to_sv

det = load_standard_detector(backend="ultralytics")   # YOLOv8x, imgsz1280, conf0.15
det.load()

TRACK = sv.ColorLookup.TRACK
box_ann   = sv.RoundBoxAnnotator(color_lookup=TRACK, thickness=2)                 # MÀU box theo track
label_ann = sv.LabelAnnotator(color_lookup=TRACK, text_scale=0.5, text_thickness=1,
                              text_position=sv.Position.TOP_LEFT)                  # TÊN NHÃN
trace_ann = sv.TraceAnnotator(color_lookup=TRACK, thickness=3, trace_length=64)   # QUỸ ĐẠO
print("✅ YOLO + annotator (màu/nhãn/quỹ đạo) sẵn sàng.")

## 3) Hàm gán nhãn (màu + tên + quỹ đạo) + chạy 1 video

In [ ]:
def new_tracker():
    for kw in (dict(track_activation_threshold=0.1, minimum_consecutive_frames=1, lost_track_buffer=120),
               dict(track_thresh=0.1), {}):
        try: return sv.ByteTrack(**kw)
        except TypeError: continue
    return sv.ByteTrack()

def annotate(frame, sd):
    f = frame.copy()
    f = trace_ann.annotate(f, sd)                     # 🌀 QUỸ ĐẠO (vẽ dưới cùng)
    f = box_ann.annotate(f, sd)                       # 🎨 box MÀU theo track
    names = sd.data.get("class_name") if getattr(sd, "data", None) else None
    tids = sd.tracker_id
    labels = []
    for i in range(len(sd)):
        nm = str(names[i]) if names is not None else "obj"
        tid = int(tids[i]) if tids is not None and tids[i] is not None else "?"
        labels.append(f"{nm} #{tid}")                 # 🏷️ TÊN NHÃN: loại xe + track-id
    return label_ann.annotate(f, sd, labels=labels)

def run_annotate(video, prompt, out_mp4, reso=(1280, 720), max_frames=180):
    tr, w, h = new_tracker(), *reso
    try: sm = sv.DetectionsSmoother(length=8)
    except Exception: sm = None
    cap = cv2.VideoCapture(video)
    vw = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"), 15, reso)
    i, seen, cls = 0, set(), {}
    while i < max_frames:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, reso)
        sd = _to_sv(det.detect(fr, prompt).detections, sv, np)
        sd = tr.update_with_detections(sd)
        if sm is not None:
            try: sd = sm.update_with_detections(sd)
            except Exception: pass
        nm = sd.data.get("class_name") if getattr(sd, "data", None) else None
        if sd.tracker_id is not None:
            for j, t in enumerate(sd.tracker_id):
                if t is not None:
                    seen.add(int(t))
                    if nm is not None: cls[str(nm[j])] = cls.get(str(nm[j]), 0) + 1
        vw.write(annotate(fr, sd)); i += 1
    vw.release(); cap.release()
    return dict(frames=i, tracks=len(seen), by_class=cls)
print("✅ Hàm gán nhãn sẵn sàng.")

## 4) 2 VIDEO cho bài PHƯƠNG TIỆN + tải

In [ ]:
def dl(name):
    p = os.path.join(WORK, name)
    if not os.path.exists(p):
        os.system(f'wget -q "https://media.roboflow.com/supervision/video-examples/{name}" -O "{p}"')
    return p if os.path.exists(p) and os.path.getsize(p) > 100000 else None

VIDEOS = [
    dict(name="giao lo", path=dl("vehicles-2.mp4"), prompt="vehicle"),   # giao lộ nhiều xe
    dict(name="cao toc", path=dl("vehicles.mp4"),   prompt="vehicle"),   # cao tốc, xe chạy nhanh → quỹ đạo dài
]
for v in VIDEOS:
    print(("OK  " if v["path"] else "MISSING  "), v["name"], "→", v["path"])

## 5) ▶️ Gán nhãn cả 2 video → lưu VIDEO + ẢNH

In [ ]:
OUTDIR = os.path.join(WORK, "label_trace_out"); os.makedirs(OUTDIR, exist_ok=True)
results = []
for v in VIDEOS:
    if not v["path"]:
        print("bỏ", v["name"], "(thiếu video)"); continue
    mp4 = os.path.join(OUTDIR, f"{v['name'].replace(' ', '_')}_annot.mp4")
    print(f"\n▶ {v['name']} — gán màu+nhãn+quỹ đạo…")
    r = run_annotate(v["path"], v["prompt"], mp4)
    by = ", ".join(f"{k}:{c}" for k, c in sorted(r["by_class"].items(), key=lambda x: -x[1]))
    print(f"   {r['frames']} frame · {r['tracks']} xe khác nhau · loại: {by}")
    # ảnh 1 frame gần cuối (quỹ đạo đã dài)
    cap = cv2.VideoCapture(mp4); nn = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, nn - 2)); ok, fr = cap.read(); cap.release()
    png = mp4.replace(".mp4", ".png")
    if ok:
        cv2.imwrite(png, fr)
        plt.figure(figsize=(11, 6)); plt.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        plt.title(f"{v['name']}: {r['tracks']} xe · {by}"); plt.axis("off"); plt.show()
    results.append(dict(name=v["name"], mp4=mp4, png=png, **r))

## 6) 🎥 Xem/tải video annotate (H.264)

In [ ]:
from IPython.display import Video, display, Markdown
for r in results:
    display(Markdown(f"### {r['name']} — {r['tracks']} xe khác nhau"))
    h264 = r["mp4"].replace(".mp4", "_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", r["mp4"],
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", h264], check=False)
    display(Video(h264 if os.path.exists(h264) else r["mp4"], embed=True, width=760))
print("💾 Video + ảnh đã lưu ở:", OUTDIR)

### Ghi chú
- **Màu theo track** (`ColorLookup.TRACK`): mỗi xe một màu ổn định suốt video → dễ nhìn xe nào là xe nào.
- **Tên nhãn**: lấy từ lớp COCO của YOLO (`car`/`truck`/`bus`/`motorcycle`) + track-id.
- **Quỹ đạo** (`TraceAnnotator`): vệt nối tâm xe qua các frame — thấy hướng & đường đi. Đổi độ
  dài vệt bằng `trace_length` (Cell 2). Muốn theo màu LỚP thay vì track: đổi `ColorLookup.TRACK`
  → `ColorLookup.CLASS`. Đổi 2 video khác: sửa `VIDEOS` (Cell 4).